# 분류 성능 지표 실습

**Precision · Recall · F1 · ROC AUC · 혼동행렬**

분류 결과를 맞춘 비율과 놓친 비율의 관점에서 평가하는 지표들.

소재 분야에서 이해하기: 결함을 놓치지 않는 것이 중요하면 재현율을 우선한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 평가 지표 문서](https://scikit-learn.org/stable/modules/model_evaluation.html)

## 1. 혼동행렬에서 출발합니다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

n = 2000
features = rng.normal(0, 1, (n, 3))
defect = (1.8 * features[:, 0] + features[:, 1] - 1.5 + rng.normal(0, 1, n) > 0).astype(int)
print('결함 비율 %.1f%%' % (100 * defect.mean()))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

X_tr, X_te, y_tr, y_te = train_test_split(features, defect, test_size=0.3, random_state=0, stratify=defect)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
matrix = confusion_matrix(y_te, model.predict(X_te))
ConfusionMatrixDisplay(matrix, display_labels=['ok', 'defect']).plot(cmap='Blues')
plt.show()
tn, fp, fn, tp = matrix.ravel()
print('정밀도 = TP/(TP+FP) = %d/%d = %.3f' % (tp, tp + fp, tp / (tp + fp)))
print('재현율 = TP/(TP+FN) = %d/%d = %.3f' % (tp, tp + fn, tp / (tp + fn)))
print(classification_report(y_te, model.predict(X_te), target_names=['ok', 'defect'], digits=3))

## 2. ROC와 PR 곡선

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

probability = model.predict_proba(X_te)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
fpr, tpr, _ = roc_curve(y_te, probability)
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('false positive rate'); axes[0].set_ylabel('true positive rate')
axes[0].set_title('ROC AUC %.3f' % roc_auc_score(y_te, probability))
precision, recall, _ = precision_recall_curve(y_te, probability)
axes[1].plot(recall, precision); axes[1].axhline(y_te.mean(), color='k', ls='--')
axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision')
axes[1].set_title('average precision %.3f' % average_precision_score(y_te, probability))
plt.tight_layout(); plt.show()

## 3. 임계값은 목적에 따라 정합니다

In [ ]:
cost_miss, cost_false_alarm = 100.0, 5.0     # 결함을 놓치는 비용이 훨씬 큰 상황
for threshold in np.arange(0.1, 0.9, 0.1):
    flagged = probability > threshold
    misses = int(((~flagged) & (y_te == 1)).sum())
    false_alarms = int((flagged & (y_te == 0)).sum())
    print('임계값 %.1f -> 놓침 %3d, 오경보 %3d, 총비용 %7.0f'
          % (threshold, misses, false_alarms, misses * cost_miss + false_alarms * cost_false_alarm))

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#classification-metrics)을 여세요.